## 上下文与记忆
### 1.1 为什么需要记忆 (Memory)

In [1]:

import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务

model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}},  # 关键：永久关闭思考模式
)

In [4]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

messages1 = [
      HumanMessage(" 你好，我叫小明 "),
      AIMessage(" 很高兴认识你，小明 "),
      HumanMessage(" 你用一句话介绍下你自己 "),
]

agent = create_agent(
      model=model,
      tools=[],
)

response1 = agent.invoke({"messages": messages1})

for msg in response1["messages"]:
      msg.pretty_print()

messages2 = [HumanMessage(" 我叫什么名字？ ")]

response2 = agent.invoke({"messages": messages2})

for msg in response2["messages"]:
      msg.pretty_print()

================================ Human Message =================================

 你好，我叫小明 
================================== Ai Message ==================================

 很高兴认识你，小明
================================ Human Message =================================

 你用一句话介绍下你自己 
================================== Ai Message ==================================

我是DeepSeek，由深度求索公司创造的AI助手，很高兴为你服务！
================================ Human Message =================================

 我叫什么名字？ 
================================== Ai Message ==================================

您好！根据我们的对话记录，我目前还不知道您的名字。您还没有告诉过我。😊

如果您愿意告诉我您的名字，我很乐意记住它，并在后续对话中称呼您！有什么我可以帮您的吗？


对比

In [5]:
messages3 = [
      HumanMessage(" 你好，我叫小明 "),
      AIMessage(" 很高兴认识你，小明 "),
      HumanMessage(" 你用一句话介绍下你自己 "),
      AIMessage(
            " 我是一个由 OpenAI 训练的人工智能助手，可以帮你回答问题、写作和解决各种任务。 "
      ),
      HumanMessage(" 我叫什么名字？ "),
]

response3 = agent.invoke({"messages": messages3})

for msg in response3["messages"]:
      msg.pretty_print()

================================ Human Message =================================

 你好，我叫小明 
================================== Ai Message ==================================

 很高兴认识你，小明
================================ Human Message =================================

 你用一句话介绍下你自己 
================================== Ai Message ==================================

 我是一个由 OpenAI 训练的人工智能助手，可以帮你回答问题、写作和解决各种任务。
================================ Human Message =================================

 我叫什么名字？ 
================================== Ai Message ==================================

你叫小明。😊


### 2.1 基于内存的持久化器

In [6]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=model, tools=[])
print("\n 第一轮对话： ")
response1 = agent.invoke({"messages": [HumanMessage(" 我叫张三 ")]})
print(f"Agent: {response1['messages'][-1].content}")

print("\n 第二轮对话： ")
response2 = agent.invoke({"messages": [HumanMessage(" 我叫什么？ ")]})
print(f"Agent: {response2['messages'][-1].content}")


 第一轮对话： 
Agent: 你好，张三！很高兴认识你。😊

有什么我可以帮你的吗？无论是聊天、解答问题、还是需要一些建议，我都很乐意帮忙！

 第二轮对话： 
Agent: 你好！从我们的对话记录来看，你还没有告诉我你的名字。如果你愿意分享，我很乐意知道该怎么称呼你！😊


对比

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint

checkpointer = InMemorySaver()

# 1. 创建 Agent 时添加 checkpointer
agent = create_agent(model=model, checkpointer=checkpointer)  # 添加内存管理

# 2. 调用时指定 thread_id
config = {"configurable": {"thread_id": "1"}}

print("\n 第一轮对话： ")
response1 = agent.invoke(
      {"messages": [HumanMessage(" 我叫张三 ")]}, config=config  # 传入 config
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n 第二轮对话： ")
response2 = agent.invoke(
      {"messages": [HumanMessage(" 我叫什么？ ")]}, config=config  # 使用相同的 thread_id
)
# print(f"Agent: {response2['messages'][-1].content}")
latest_state = agent.get_state(config)
rprint(latest_state)

### **步骤 2：** 第三轮对话
当我们再次进行对话时，直接带入线程 ID ，即可带入此前对话记忆：

In [ ]:
print("\n 第三轮对话： ")
response3 = agent.invoke(
      {"messages": [HumanMessage(" 我刚才问了什么问题？ ")]},
      config=config,  # 使用相同的 thread_id
)
latest_state = agent.get_state(config)
rprint(latest_state)

### **步骤 3：** 更新线程 ID
而如果更新线程 ID ，则会重新开启对话：

In [13]:
config2 = {"configurable": {"thread_id": "2"}}

response4 = agent.invoke(
      {"messages": [HumanMessage(" 你还记得我叫什么名字么？ ")]}, config=config2
)

print(response4['messages'][-1].content)

你好！很抱歉，我无法记住你的名字。每次对话开始时，我都是从一个全新的状态开始的，没有之前对话的记忆。这是为了保护用户的隐私和安全。

如果你愿意告诉我你的名字，我很乐意在这次对话中用名字称呼你！有什么我可以帮你的吗？😊


### 2.1.3 关键步骤说明
第 1 步：初始化记忆引擎

In [16]:
# 会话 1
config1 = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": [HumanMessage("我叫张三")]}, config=config1)

# 会话 2
config2 = {"configurable": {"thread_id": "2"}}
resp = agent.invoke({"messages": [HumanMessage("我叫什么？")]}, config=config2)
# 断言：resp 里不应该出现“张三” → 验证隔离生效

### **场景 1：** 多用户聊天

In [19]:
agent = create_agent(
      model=model,
      tools=[],
      checkpointer=InMemorySaver()
)
from langchain_core.messages import HumanMessage

# 用户 Alice
config_alice = {"configurable": {"thread_id": "user_alice"}}
agent.invoke(
      {"messages": [HumanMessage(content="你好，我是 Alice")]},  # 真实的消息列表
      config=config_alice
)

# 用户 Bob
config_bob = {"configurable": {"thread_id": "user_bob"}}
agent.invoke(
      {"messages": [HumanMessage(content="你好，我是 Bob")]},  # 真实的消息列表
      config=config_bob
)

# 两个会话完全独立

{'messages': [HumanMessage(content='你好，我是 Bob', additional_kwargs={}, response_metadata={}, id='ce344228-c949-4872-830d-d8867d9d0a58'),
  AIMessage(content='你好，Bob！很高兴认识你。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 8, 'total_tokens': 22, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'd3d8e5a0-68cc-441a-b3cc-97d7ac365d6e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ea25-735b-77f9-8f6e-f29e1c06e38b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 14, 'total_tokens': 22, 'input_token_details': {'cache_read': 0}, 'output_token_details'

### **场景 2：** 同一用户的不同任务

In [30]:
from langchain_core.messages import HumanMessage

# ==================== Alice 的任务 1：旅行规划 ====================
config_task1 = {"configurable": {"thread_id": "alice_task_travel"}}

agent.invoke(
      {"messages": [HumanMessage(content="我要去日本旅行，预算 1 万")]},
      config=config_task1,
)
r1 = agent.invoke(
      {"messages": [HumanMessage(content="我的预算是多少？有什么建议")]},
      config=config_task1,
)
travel_text = "".join(m.content for m in r1["messages"])
print("【任务1-旅行】", r1["messages"][-1].content)

# ==================== Alice 的任务 2：改简历 ====================
config_task2 = {"configurable": {"thread_id": "alice_task_resume"}}

agent.invoke(
      {"messages": [HumanMessage(content="帮我改简历，我是做后端的")]},
      config=config_task2,
)
r2 = agent.invoke(
      {"messages": [HumanMessage(content="我是做什么方向的？")]},
      config=config_task2,
)
resume_text = "".join(m.content for m in r2["messages"])
print("【任务2-简历】", r2["messages"][-1].content)

print("\n🎉 同一用户的不同任务完全隔离，且各自记忆保持")

【任务1-旅行】 你的预算是 **1 万元（人民币）**。

针对这个预算去日本旅行，给你几点核心建议：

1. **机票优先控制**：提前1–2个月订票，选廉价航空或转机航班，尽量把往返机票压在 **2500–3500元**。
2. **住宿选经济型**：青旅、胶囊旅馆或民宿拼房，控制在 **150–300元/晚**，优先选地铁站附近。
3. **跨城用夜巴**：大阪—东京等长途可选夜行巴士，既省交通费又省一晚住宿。
4. **餐饮靠便利店+定食**：早餐便利店，午晚餐选松屋、吉野家、拉面店，每餐 **30–60元**。
5. **景点多选免费的**：伏见稻荷、浅草寺、奈良公园、东京都厅展望台等都免费，付费景点选1–2个即可。
6. **避开旺季**：樱花季、盂兰盆节、国庆期间住宿和机票会大幅涨价。
7. **少购物、留应急现金**：购物控制在500–1000元，另备3–5万日元现金应急。

**行程建议**：6–8天，东京+大阪+京都（或选一个区域深度游），总花费可控制在 **8000–10000元**。

需要我帮你做一版**严格1万元以内的详细行程和预算表**吗？
【任务2-简历】 我得坦诚地跟你说：**这个问题，我们已经在原地打转了七八轮了。** 😅

规则很简单——我能看见的，**只有你在对话框里打出来的字**。而你这几轮打的字加起来就一句话的意思："我是做后端的。" 就凭这个，我判断不出方向，而且我不愿意为了显得"有用"而瞎编一个方向给你，那才是真的坑你。

---

## 我们换个方式，这次一定能破局

**别再问"我是什么方向"了，直接做下面任意一件事：**

**① 把简历文字粘过来**（Ctrl+A → Ctrl+C → Ctrl+V，最简单）
**② 或者对着填空回一句：**

```
语言：______
年限：______
最近一个项目：______（一句话说干啥的）
想投的岗位：______
```

**③ 实在不想打字，那就回一个词也行**，比如：Java / Go / 电商 / 支付 / 几年。

---

只要收到上面**任何一条具体信息**，我立刻给你：
- 方向定位（业务后端？高并发？中间件？）
- 简历逐条重写 + 量化 + 技术亮点
- 该删的空话、该突出的重点

**但在收到具体信息前，我这边是真的没法产出东西的。** 